In [48]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [49]:
#   Load The Dataset:
df=pd.read_csv('data/Titanic-Dataset.csv')
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Cabin: ~77% missing -> Drop the column 

Age: ~20% missing -> Fill using Median 

Embarked: sirf 2 missing -> Fill using Mode 

In [50]:
df=df.drop(columns='Cabin').copy()
df.shape

(891, 11)

## Splitting:

In [51]:
x=df.drop(columns='Survived').copy()
y=df['Survived']

In [52]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2,stratify=y)

# Missing Value Imputation

In [53]:
impute=SimpleImputer(strategy='median')
x_train[['Age']]=impute.fit_transform(x_train[['Age']])
x_test[['Age']]=impute.transform(x_test[['Age']])

In [54]:
impute=SimpleImputer(strategy='most_frequent')
x_train[['Embarked']]=impute.fit_transform(x_train[['Embarked']])
x_test[['Embarked']]=impute.transform(x_test[['Embarked']])

In [55]:
print(x_train.isnull().sum().sum())
print(x_test.isnull().sum().sum())

0
0


#   Encoding Categories

In [56]:
#   Removing useless columns:
x_train=x_train.drop(columns=['PassengerId', 'Name', 'Ticket']).copy()
x_test=x_test.drop(columns=['PassengerId', 'Name', 'Ticket']).copy()

In [57]:
x_train['Sex'].value_counts()

Sex
male      459
female    253
Name: count, dtype: int64

In [58]:
ct = ColumnTransformer(
    [('ohe', OneHotEncoder(drop='first', sparse_output=False), ['Sex', 'Embarked'])],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

In [59]:
x_train.head()

,Sex_male,Embarked_Q,Embarked_S,Pclass,Age,SibSp,Parch,Fare
692,1.0,0.0,1.0,3,28.5,0,0,56.4958
481,1.0,0.0,1.0,2,28.5,0,0,0.0000
527,1.0,0.0,1.0,1,28.5,0,0,221.7792
855,0.0,0.0,1.0,3,18.0,0,1,9.3500
801,0.0,0.0,1.0,2,31.0,1,1,26.2500


In [61]:
x_train.dtypes

Sex_male      float64
Embarked_Q    float64
Embarked_S    float64
Pclass          int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
dtype: object

In [63]:
x_train.to_csv('data/x_train.csv')
y_train.to_csv('data/y_train.csv')
x_test.to_csv('data/x_test.csv')
y_test.to_csv('data/y_test.csv')